In [1]:
!hostname

node108


In [2]:

import numpy as np 
import pandas as pd 
from pathlib import Path
import pickle 
import IPython.display as ipd
# import whisper


# Demo for getting transcripts from SpokenWikipediaCorpus stimuli used in 2023 mono word recognition experiment

### Note 11/2023: Actual transcription done using WHISPER to transcribe audio with script `src/get_swc_prolific_manifest_transcripts.py`

Foregrounds and cues live in manifest:
`/om/user/imgriff/datasets/human_word_rec_SWC_2023/unique_cue_and_target_manifest_for_human_expmnt.pdpkl`

Stim live at `/om/user/imgriff/datasets/human_word_rec_SWC_2023`


## Note on source transcripts: 
Original SWC transcription is fuzzy, and missed a few words. Word labels in `/om2/user/msaddler/spatial_audio_pipeline/assets/swc/manifest_all_words.pdpkl` are curated set from larger manifest `/scratch2/weka/mcdermott/raygon/projects/public/jsinDataset/assets/data/interim/swc/mungedFinalDataframeWords_swc_readerNormalized_sexNormalized_accent.pdpkl`.      

This second manifest is itself a distilled version of SWC transcripts. Backing out to the orginal manifests beyond this point requires re-cutting the original transcripts. 

EG for getting manifest info form larger manifest:

In [5]:

'''
all_swc_manifest = pd.read_pickle('/scratch2/weka/mcdermott/raygon/projects/public/jsinDataset/assets/data/interim/swc/mungedFinalDataframeWords_swc_readerNormalized_sexNormalized_accent.pdpkl')
curated_swc_manifest = pd.read_pickle('/om2/user/msaddler/spatial_audio_pipeline/assets/swc/manifest_all_words.pdpkl')
## Get full file names of target source files
unique_manifest['target_src_fn'] = curated_swc_manifest.loc[unique_manifest.src_ix.values, 'src_fn'].values
# #get just the base name of the article from the full file name
unique_manifest['target_article_name'] = unique_manifest['target_src_fn'].str.split('/validArticles/').str[-1].str.split('/').str[0]
# get article name from full file name for all source manifest
all_swc_manifest['article_names'] = all_swc_manifest['path'].str.split('/validArticles/').str[-1].str.split('/').str[0]
# find all rows in all_swc_manifest that have the same source file name that begin within 2 seconds of the start and end times in the unique manifest

eg_stim_row = unique_manifest.iloc[0]
all_swc_manifest.loc[(all_swc_manifest['article_names'] == eg_stim_row.target_article_name) 
                    & (all_swc_manifest['start_ms'].astype('int') >= (eg_stim_row.clip_start_in_s - 2)*1000) 
                    & (all_swc_manifest['end_ms'].astype('int') <= (eg_stim_row.clip_end_in_s + 2)*1000)]
''' 

"\nall_swc_manifest = pd.read_pickle('/scratch2/weka/mcdermott/raygon/projects/public/jsinDataset/assets/data/interim/swc/mungedFinalDataframeWords_swc_readerNormalized_sexNormalized_accent.pdpkl')\ncurated_swc_manifest = pd.read_pickle('/om2/user/msaddler/spatial_audio_pipeline/assets/swc/manifest_all_words.pdpkl')\n## Get full file names of target source files\nunique_manifest['target_src_fn'] = curated_swc_manifest.loc[unique_manifest.src_ix.values, 'src_fn'].values\n# #get just the base name of the article from the full file name\nunique_manifest['target_article_name'] = unique_manifest['target_src_fn'].str.split('/validArticles/').str[-1].str.split('/').str[0]\n# get article name from full file name for all source manifest\nall_swc_manifest['article_names'] = all_swc_manifest['path'].str.split('/validArticles/').str[-1].str.split('/').str[0]\n# find all rows in all_swc_manifest that have the same source file name that begin within 2 seconds of the start and end times in the unique

In [3]:
unique_manifest = pd.read_pickle("/om/user/imgriff/datasets/human_word_rec_SWC_2023/unique_cue_and_target_manifest_for_human_expmnt.pdpkl")
word_dict = pickle.load(open("/om2/user/imgriff/datasets/commonvoice_9/en/cv_800_word_label_to_int_dict.pkl", 'rb'))


In [4]:
# This buys us nothing more than curated version all_swc_manifest = pd.read_pickle('/scratch2/weka/mcdermott/raygon/projects/public/jsinDataset/assets/data/interim/swc/mungedFinalDataframeWords_swc_readerNormalized_sexNormalized_accent.pdpkl')
curated_swc_manifest = pd.read_pickle('/om2/user/msaddler/spatial_audio_pipeline/assets/swc/manifest_all_words.pdpkl')

## Example on getting context transcripts 

In [5]:
## Get full file names of target source files
eg_ix = 12

eg_stim_row = curated_swc_manifest.iloc[unique_manifest.loc[eg_ix,'src_ix']]
print(eg_stim_row)
ipd.display(ipd.Audio(unique_manifest.loc[eg_ix,'src_fn']))  
context_buffer = (2 - eg_stim_row.clip_dur_in_s)  # 
curated_swc_manifest.loc[(curated_swc_manifest['src_fn'] == eg_stim_row.src_fn) 
                    & (curated_swc_manifest['clip_start_in_s'].astype('int') >= (eg_stim_row.clip_start_in_s - context_buffer)) 
                    & (curated_swc_manifest['clip_end_in_s'].astype('int') <= (eg_stim_row.clip_end_in_s + context_buffer))].word.to_list()

client_id                                                          aguerriero
clip_dur_in_s                                                             0.4
clip_end_in_s                                                          230.24
clip_start_in_s                                                        229.84
corpus                                                                    swc
gender                                                                   male
gender_int                                                                  1
split                                                                     NaN
split_int                                                                   0
sr                                                                      44100
src_fn                      /scratch2/weka/mcdermott/msaddler/swc/english/...
total_file_duration_in_s                                          1464.280816
word                                                            

['played', 'with', 'nice', 'tight']

In [6]:
## Get single-distractor conditions 

import re
# get cond name map
map_path = Path('/om2/user/imgriff/projects/Auditory-Attention/human_saddler_attn_expmt_cond_map.pkl')
with open(map_path, 'rb') as handle:
    stim_cond_map = pickle.load(handle)

stim_cond_map = {f"condition_{k:02}": v for k,v in stim_cond_map.items()}

parent_dir = Path("/om/user/imgriff/datasets/human_word_rec_SWC_2023/sounds/")
wanted_conditions = [cond_dir for cond_dir, cond_tuple in stim_cond_map.items() if cond_tuple[0] == '1-talker']
single_distractor_manifests = pd.concat([pd.read_pickle(parent_dir / cond_dir / 'manifest.pdpkl') for cond_dir in wanted_conditions], axis=0, ignore_index=True)
single_distractor_manifests["base_str"] = single_distractor_manifests.mixture_fn.apply(lambda x: f"stim/{x.as_posix().split('sounds/')[1]}") 
single_distractor_manifests.head()

,target_sr,experiment_key_target_word_ix,cue_sr,target_fn,cue_fn,word,word_int,condition,snr,src_ix,client_id,target_gender,target_f0,distractor_fn,distractor_f0,distractor_gender,distractor_word,mixture_fn,base_str
0,44100,0,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,/om/user/imgriff/datasets/spatial_audio_pipeli...,above,1,1-talker,-9,310886,mangst,male,109.965641,/om/user/imgriff/datasets/spatial_audio_pipeli...,125.085843,male,countries,/om/user/imgriff/datasets/human_word_rec_SWC_2...,stim/condition_30/000.wav
1,44100,1,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,/om/user/imgriff/datasets/spatial_audio_pipeli...,according,3,1-talker,-9,181635,the-voice-of-hassocks,male,117.046969,/om/user/imgriff/datasets/spatial_audio_pipeli...,135.868215,male,studio,/om/user/imgriff/datasets/human_word_rec_SWC_2...,stim/condition_30/001.wav
2,44100,2,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,/om/user/imgriff/datasets/spatial_audio_pipeli...,across,4,1-talker,-9,310319,mangst,male,112.441986,/om/user/imgriff/datasets/spatial_audio_pipeli...,120.255924,male,married,/om/user/imgriff/datasets/human_word_rec_SWC_2...,stim/condition_30/002.wav
3,44100,3,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,/om/user/imgriff/datasets/spatial_audio_pipeli...,action,5,1-talker,-9,937425,preslethe,male,114.925214,/om/user/imgriff/datasets/spatial_audio_pipeli...,183.153323,male,military,/om/user/imgriff/datasets/human_word_rec_SWC_2...,stim/condition_30/003.wav
4,44100,4,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,/om/user/imgriff/datasets/spatial_audio_pipeli...,activities,7,1-talker,-9,940458,vaslittlecrow,female,207.809580,/om/user/imgriff/datasets/spatial_audio_pipeli...,125.965637,male,worked,/om/user/imgriff/datasets/human_word_rec_SWC_2...,stim/condition_30/004.wav


In [7]:
ipd.Audio(single_distractor_manifests.target_fn[0])

### Get transcripts for target and distractor sentences and append to df

In [8]:
from tqdm.auto import tqdm


/weka/scratch/weka/mcdermott/imgriff/conda_envs/pytorch_2_whisper/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
target_fns = single_distractor_manifests.target_fn.to_list()
distractor_fns = single_distractor_manifests.distractor_fn.to_list()

model = whisper.load_model("medium").cuda()
options = whisper.DecodingOptions()

batch_size = 160
n_iters = len(target_fns) // batch_size

transcripts = []
for iter in tqdm(range(n_iters)):
    target_batch = target_fns[iter*batch_size:(iter+1)*batch_size]
    distractor_batch = distractor_fns[iter*batch_size:(iter+1)*batch_size]
    # decode the audio
    target_audio = np.array([whisper.pad_or_trim(whisper.load_audio(fname)) for fname in target_batch])
    target_mel_specs = whisper.log_mel_spectrogram(target_audio).to(model.device)

    # distrctor batch 
    distractor_audio = np.array([whisper.pad_or_trim(whisper.load_audio(fname)) for fname in distractor_batch])
    distractor_mel_specs = whisper.log_mel_spectrogram(distractor_audio).to(model.device)

    # decode the audio
    target_results = whisper.decode(model, target_mel_specs, options)
    distracted_results = whisper.decode(model, distractor_mel_specs, options)

    batch_dict = {'target_fn': target_batch, 'distractor_fn': distractor_batch,
                 'target_transcripts': [[result.text] for result in target_results],
                'distractor_transcripts': [[result.text] for result in distracted_results]}
    break
    transcripts.append(batch_dict)


  0%|          | 0/11 [02:43<?, ?it/s]


In [12]:
from collections.abc import Iterable
from num2words import num2words

def flatten(xs):
    for x in xs:
        if isinstance(x, Iterable) and not isinstance(x, (str, bytes)):
            yield from flatten(x)
        else:
            yield x

def convert_transcript(tscrpt):
    tscrpt = tscrpt.lower()
    tscrpt = re.sub(r'[^a-z0-9 ]+', '', tscrpt)
    tscrpt = [num2words(token).split('-') if token.isdigit() else token for token in tscrpt.split(' ')]
    return list(flatten(tscrpt))



[convert_transcript(result.text) for result in target_results]

[['see', 'particles', 'above', 'enceladuss'],
 ['which', 'according', 'to', 'the', 'constitution'],
 ['bands', 'cutting', 'across', 'crater', 'terrain'],
 ['so', 'urgent', 'was', 'the', 'action', 'that', 'the', 'wickham', 'abbey'],
 ['enjoyed', 'activities', 'as', 'well'],
 ['the', 'actual', 'activity', 'of', 'the', 'season'],
 ['have', 'added', 'the', 'capability'],
 ['in', 'addition', 'the', 'association', 'of', 'the'],
 ['as', 'additional', 'evidence', 'for', 'the'],
 ['relocated', 'again', 'to', 'the', 'florida'],
 ['military', 'involvement', 'against', 'the', 'common', 'spanish'],
 ['control', 'and', 'is', 'allowed', 'to', 'wander', 'the'],
 ['became', 'almost', 'taboo'],
 ['religious', 'critics', 'alone', 'faulted', 'the', 'philosophy'],
 ['clone', 'formed', 'along', 'an', 'approaching'],
 ['zappa', 'had', 'already', 'moved', 'to', 'his', 'own'],
 ['time', 'although', 'it', 'is', 'distributed'],
 ['slaves', 'in', 'america'],
 ['in', 'south', 'american', 'countries', 'and'],
 ['ha

In [31]:
all_transcripts_df = pd.concat([pd.DataFrame.from_dict(batch_dict) for batch_dict in transcripts], axis=0, ignore_index=True)

In [32]:
all_transcripts_df['target_transcript'] = all_transcripts_df['target_transcript'].apply(lambda x: x[0].lower().split(' '))
all_transcripts_df['distractor_transcript'] = all_transcripts_df['distractor_transcript'].apply(lambda x: x[0].lower().split(' '))

In [35]:
# merge back into single distractor manifest
ground_truth_single_dist_df = single_distractor_manifests.merge(all_transcripts_df, on=['target_fn', 'distractor_fn'])

In [38]:
!pwd
# updatae names target_reesults to target_transcript
ground_truth_single_dist_df.rename(columns={'target_results': 'target_transcript', 'distracted_results': 'distractor_transcript'}, inplace=True)

/om2/user/imgriff/projects/torch_2_aud_attn


In [39]:
ground_truth_single_dist_df.to_pickle("swc_prolific_2023_single_distractor_ground_truth_manifest.pdpkl")

In [4]:
# from num2words import num2words
import pandas as pd 

In [12]:
## Look at output from py script using whisper

manifest_paths = sorted(list(Path("/om/user/imgriff/datasets/human_word_rec_SWC_2023/sounds").glob("*/manifest_w_transcripts.pdpkl")))

In [28]:
## Try to merge all 

all_dfs = pd.concat([pd.read_pickle(path) for path in manifest_paths], ignore_index=True)
all_dfs.rename(columns={'target_transcripts': 'target_transcript', 'distracted_transcripts': 'distractor_transcript'}, inplace=True)

# add base_str column
all_dfs["base_str"] = all_dfs.mixture_fn.apply(lambda x: f"stim/{x.as_posix().split('sounds/')[1]}") 
all_dfs.head()

,target_sr,experiment_key_target_word_ix,cue_sr,target_fn,cue_fn,word,word_int,condition,snr,src_ix,...,target_gender,target_f0,mixture_fn,target_transcript,distractor_fn,distractor_f0,distractor_gender,distractor_word,distractor_transcript,base_str
0,44100,0,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,/om/user/imgriff/datasets/spatial_audio_pipeli...,above,1,background_musdb18hq,-9,703492,...,female,163.753088,/om/user/imgriff/datasets/human_word_rec_SWC_2...,"[extend, from, above, the, navel, to, below]",NaN,NaN,NaN,NaN,NaN,stim/condition_00/000.wav
1,44100,1,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,/om/user/imgriff/datasets/spatial_audio_pipeli...,according,3,background_musdb18hq,-9,181635,...,male,117.046969,/om/user/imgriff/datasets/human_word_rec_SWC_2...,"[which, according, to, the, constitution]",NaN,NaN,NaN,NaN,NaN,stim/condition_00/001.wav
2,44100,2,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,/om/user/imgriff/datasets/spatial_audio_pipeli...,across,4,background_musdb18hq,-9,310319,...,male,112.441986,/om/user/imgriff/datasets/human_word_rec_SWC_2...,"[bands, cutting, across, crater, terrain]",NaN,NaN,NaN,NaN,NaN,stim/condition_00/002.wav
3,44100,3,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,/om/user/imgriff/datasets/spatial_audio_pipeli...,action,5,background_musdb18hq,-9,937425,...,male,114.925214,/om/user/imgriff/datasets/human_word_rec_SWC_2...,"[so, urgent, was, the, action, that, the, wick...",NaN,NaN,NaN,NaN,NaN,stim/condition_00/003.wav
4,44100,4,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,/om/user/imgriff/datasets/spatial_audio_pipeli...,activities,7,background_musdb18hq,-9,84925,...,male,108.427679,/om/user/imgriff/datasets/human_word_rec_SWC_2...,"[special, activities, division]",NaN,NaN,NaN,NaN,NaN,stim/condition_00/004.wav


In [31]:
all_dfs.to_pickle("swc_prolific_2023_all_condition_ground_truth_manifest.pdpkl")

In [26]:
transcripts_w_digits = [t for t in all_dfs['target_transcript'].values if any([i.isdigit() for i in t])]
len(transcripts_w_digits)

0

### Look at single-distractor df 

In [ ]:
df = pd.read_pickle('swc_prolific_2023_single_distractor_ground_truth_manifest.pdpkl')

In [ ]:
df['target_transcript'][0]

['see', 'particles', 'above', "enceladus's"]

In [24]:
# check if there are digits in df['target_transcript']
transcripts_w_digits = [t for t in df['target_transcript'].values if any([i.isdigit() for i in t])]

In [40]:
# flatten nested lists in list
# https://stackoverflow.com/questions/952914/how-to-make-a-flat-list-out-of-list-of-lists
nested_list = [num2words(token).split('-') if token.isdigit() else token for token in transcripts_w_digits[0]]

# flattten nested list of strings - if element is a list, unpack it, else keep the string 


from collections.abc import Iterable

def flatten(xs):
    for x in xs:
        if isinstance(x, Iterable) and not isinstance(x, (str, bytes)):
            yield from flatten(x)
        else:
            yield x

def convert_transcript(tscrpt):
    return list(flatten([num2words(token).split('-') if token.isdigit() else token for token in tscrpt]))

df['target_transcript'] = df['target_transcript'].apply(convert_transcript)
df['distractor_transcript'] = df['distractor_transcript'].apply(convert_transcript)


In [42]:
print(len([t for t in df['target_transcript'].values if any([i.isdigit() for i in t])]))
print(len([t for t in df['distractor_transcript'].values if any([i.isdigit() for i in t])]))

0
0


In [43]:
df.to_pickle("swc_prolific_2023_single_distractor_ground_truth_manifest.pdpkl")